# Inventing a Recommender System

How does Netflix know what you should watch next? How does Spotify build your
Discover Weekly? Nobody at those companies watches movies on your behalf and
takes notes. The trick is beautifully simple:

> **Find people whose taste matches yours, and recommend what *they* loved
> and you haven't seen yet.**

This is called **collaborative filtering**, and in this chapter you will invent
it from scratch — using *real* movie ratings collected from a live class of
students. By the end you will compute, for any person in the class, the three
movies they are most likely to enjoy.

The plan:

1. **Clean the data** — real data is messy: missing ratings, people who barely
   rated anything, generous vs stingy raters.
2. **Measure taste similarity** — invent cosine similarity, then compute *every*
   pair of similarities in a single matrix multiplication.
3. **Recommend** — combine similarity with ratings to score unseen movies.
4. **Bonus** — how would this scale to Netflix size? Invent the MapReduce trick
   for multiplying gigantic sparse matrices.

You'll need `pandas` and `numpy`. If you know a little pandas (reading a CSV,
selecting columns) you're ready; we'll introduce everything else as we go.

## The data

Run the cell below as-is. It loads `movies_ratings.csv` — 25 Bollywood movies
rated 0–5 by 18 people (a real class!). Rows are movies, columns are people,
and an empty cell means *that person never rated that movie*. Pandas shows
these as `NaN` (Not a Number).

In [ ]:
# Provided — run this cell as-is.
import pandas as pd
import numpy as np

try:
    df = pd.read_csv("movies_ratings.csv", index_col="Movies_Name")
except FileNotFoundError:
    url = ("https://raw.githubusercontent.com/cloudxlab/learnbyinventing/"
           "main/recommender/movies_ratings.csv")
    df = pd.read_csv(url, index_col="Movies_Name")

print("shape (movies, people):", df.shape)
df.head(8)

## Part 1 — Cleaning real data

### Exercise 1.1 — Drop people who barely rated anything

Look at the output above: the table is full of `NaN`. Some people rated almost
every movie; a few rated *nothing at all*. Someone with 0 or 2 ratings tells us
nothing about their taste — keeping them would only add noise.

**Your task:** build a DataFrame called `ratings` that keeps only the people
(columns) who rated **at least 3 movies**.

Hints:
- `df.count()` gives the number of *non-NaN* values in each column. Print it —
  which three people rated nothing?
- `df.drop(columns=[...])` returns a copy without those columns, or select the
  columns you want to keep with `df[list_of_names]`.

In [ ]:
ratings = ...   # replace this

# Tests
assert ratings.shape == (25, 15)              # 18 people -> 15
assert "Yukta" not in ratings.columns         # 0 ratings
assert "SandeepGiri" in ratings.columns       # 15 ratings — stays
print("Kept:", list(ratings.columns))

### Exercise 1.2 — Generous raters vs stingy raters

Print `ratings.mean().sort_values()`. Notice something? Bhaskar's *average*
rating is about **1.7** while Venu's is about **4.1**. Does Venu love every
movie and Bhaskar hate them all? More likely they just use the scale
differently — a 3 from Bhaskar might mean the same as a 5 from Venu.

If we compare raw numbers, the stingy and generous raters will look like they
disagree even when their *taste* is identical. We need to put everyone on the
same scale.

**The fix — min-max scaling, per person:** stretch each person's ratings so
their lowest becomes 0 and their highest becomes 1:

$$\text{scaled} = \frac{x - \min}{\max - \min}$$

**Your task:** create `scaled` from `ratings` using the formula above, applied
column-by-column.

Hints:
- pandas broadcasting does the per-column work for you:
  `ratings - ratings.min()` subtracts each column's min from that column.
- One worry: what if someone gave *every* movie the same rating? Then
  `max - min = 0` and we'd divide by zero. Check first:
  `(ratings.max() > ratings.min()).all()` — you'll find this data is safe.
- `NaN` stays `NaN` through arithmetic — exactly what we want. We haven't
  invented ratings for anyone; we've only rescaled the ones that exist.

In [ ]:
scaled = ...   # replace this

# Tests
assert np.allclose(scaled.min(), 0)           # every person's lowest -> 0
assert np.allclose(scaled.max(), 1)           # every person's highest -> 1
assert scaled.isna().sum().sum() == ratings.isna().sum().sum()   # NaNs untouched
print("NaNs remaining:", int(scaled.isna().sum().sum()))

### Exercise 1.3 — Filling the holes

The similarity math coming in Part 2 can't handle `NaN` — we need a number in
every cell. But which number?

Think about the options before reading on:
- Fill with **0**? That says "this person would *hate* this movie" — a strong
  claim we have no evidence for.
- Fill with **1**? Same problem, opposite direction.
- Fill with **that person's average scaled rating**? That says "no opinion —
  assume they'd feel about it the way they feel about a typical movie."
  Neutral. This is the one.

**Your task:** create `filled` from `scaled`, replacing each `NaN` with the
*mean of that person's column*.

Hints:
- `scaled.mean()` gives the per-column means (pandas skips NaN automatically).
- `scaled.fillna(...)` accepts a Series of per-column values.

In [ ]:
filled = ...   # replace this

# Tests
assert filled.isna().sum().sum() == 0
# SandeepGiri never rated Mughal-e-Azam, so that cell must be his scaled mean:
assert abs(filled.loc["Mughal-e-Azam", "SandeepGiri"] - 0.3538) < 1e-3
print(filled.head(5).round(2))

## Part 2 — Measuring taste

### Exercise 2.1 — Cosine similarity

Each person is now a column of 25 numbers — a **vector** in 25-dimensional
space (one dimension per movie). Two people with similar taste should have
vectors pointing in a similar *direction*.

In the Ancient Secrets of Prediction chapter you met the dot product. Recall
the key fact:

$$\vec{a} \cdot \vec{b} = |\vec{a}|\,|\vec{b}|\,\cos\theta$$

where $\theta$ is the angle between the vectors. Rearranging:

$$\cos\theta = \frac{\vec{a} \cdot \vec{b}}{|\vec{a}|\,|\vec{b}|}$$

This is **cosine similarity**: 1 means the vectors point the same way
(identical taste), 0 means perpendicular (unrelated taste). Because it divides
away the lengths, it measures *direction only* — one more defense against
scale differences between raters.

**Your task:** write `cosine_similarity(a, b)` for two columns of `filled`.

Hints:
- Dot product: `(a * b).sum()`.
- Length of a vector: `np.sqrt((a ** 2).sum())`.

**Write your solution in `exercises/ex_2_1_cosine_similarity.py`**

In [ ]:
def cosine_similarity(a, b):
    pass   # replace this

# Tests
me = filled["SandeepGiri"]
assert abs(cosine_similarity(me, me) - 1.0) < 1e-9        # identical vectors
assert abs(cosine_similarity(me, filled["Abhishek"]) - 0.8198) < 1e-3
print("sim(SandeepGiri, Abhishek) =",
      round(cosine_similarity(me, filled["Abhishek"]), 4))

### Exercise 2.2 — Unit vectors

We want the similarity between *every* pair of people — that's
$15 \times 15 = 225$ numbers. Calling `cosine_similarity` in a double loop
would work, but there is a spectacular shortcut. It starts with one
observation:

> If every column already has length 1, cosine similarity is *just the dot
> product* — the division by lengths does nothing.

A vector divided by its own length has length 1 (a **unit vector**).

**Your task:** build the numpy array `uM` where every column of
`M = filled.values` is scaled to length 1.

Hints:
- `M = filled.values` gives a 25×15 numpy array.
- Column lengths: `np.sqrt((M ** 2).sum(axis=0))` — `axis=0` means "sum down
  each column", giving 15 lengths.
- `M / lengths` divides each column by its own length (numpy broadcasting
  matches the 15 lengths to the 15 columns).

In [ ]:
M = filled.values
uM = ...   # replace this

# Tests
assert uM.shape == (25, 15)
assert np.allclose((uM ** 2).sum(axis=0), 1)   # every column has length 1
print("all 15 columns are unit vectors")

### Exercise 2.3 — All 225 similarities in one multiplication

Here's the payoff. Remember how matrix multiplication works: entry $(i, j)$ of
$A B$ is the dot product of **row $i$ of $A$** with **column $j$ of $B$**.

So what is $uM^T uM$? Row $i$ of $uM^T$ is *column $i$ of $uM$* — person $i$'s
unit vector. So entry $(i, j)$ is the dot product of person $i$'s unit vector
with person $j$'s unit vector… which is exactly their cosine similarity. **One
matrix multiplication computes the entire similarity table.** This is not a
toy trick — it is precisely how real recommender systems (and the attention
layers inside LLMs!) compute all-pairs similarity.

**Your task:** compute `S = uM.T @ uM`, then wrap it as a labeled DataFrame:

```python
sim = pd.DataFrame(S, index=filled.columns, columns=filled.columns)
```

Before running the tests, predict: what should the diagonal of `S` be, and
why? Should `S` equal its own transpose?

In [ ]:
S = ...   # replace this
sim = ...  # replace this (labeled DataFrame)

# Tests
assert S.shape == (15, 15)
assert np.allclose(np.diag(S), 1)   # everyone matches themselves perfectly
assert np.allclose(S, S.T)          # sim(a, b) == sim(b, a)
assert abs(sim.loc["SandeepGiri", "Abhishek"] - 0.8198) < 1e-3
sim.round(2)

### Exercise 2.4 — Your taste twin

**Your task:** write `most_similar(person)` returning the name of the person
with the highest similarity to `person` — *excluding* the person themselves
(everyone's best match would otherwise be themselves, similarity 1.0).

Hints:
- `sim[person]` is a Series of similarities, indexed by name.
- `.drop(person)` removes the self-entry; `.idxmax()` returns the index label
  (the name) of the largest value.

**Write your solution in `exercises/ex_2_4_most_similar.py`**

In [ ]:
def most_similar(person):
    pass   # replace this

# Tests
assert most_similar("SandeepGiri") == "Madhuri"
for p in ["SandeepGiri", "Abhishek", "Venu", "Gurpreet"]:
    print(p, "->", most_similar(p),
          round(sim[p].drop(p).max(), 3))

## Part 3 — Making recommendations

### Exercise 3.1 — What haven't you seen?

We only want to recommend movies the person *hasn't rated yet* — no point
suggesting a movie they already gave 5 stars.

Careful: which table remembers what was actually rated? Not `filled` — we
erased that information when we filled the NaNs. Go back to `ratings`.

**Your task:** write `unrated_movies(person)` returning a list of movie names
the person never rated.

Hint: `ratings[person].isna()` is a boolean Series;
`ratings.index[boolean_series]` selects the matching movie names.

**Write your solution in `exercises/ex_3_1_unrated_movies.py`**

In [ ]:
def unrated_movies(person):
    pass   # replace this

# Tests
mine = unrated_movies("SandeepGiri")
assert len(mine) == 10
assert "Dangal" in mine and "Mughal-e-Azam" in mine
print("SandeepGiri hasn't rated:", mine)

### Exercise 3.2 — The recommender

Now assemble the pieces. To score a candidate movie $m$ for a person $p$:

$$\text{score}(m, p) = \sum_{\text{person } q} \text{rating}(m, q) \cdot \text{sim}(p, q)$$

In words: everyone in the class votes with their (filled, scaled) rating of
$m$, but people whose taste matches $p$'s get a louder voice. A movie loved by
your taste twins scores high; a movie loved only by people you disagree with
scores low.

Look closely — that sum is a dot product: the movie's row of `filled` dotted
with $p$'s column of `sim`. So *all* candidate scores at once is:

```python
scores = filled.loc[candidates] @ sim[person]
```

**Your task:** write `recommend(person, k=3)` that scores every unrated movie
and returns the `k` movie names with the highest scores, best first.

Hint: `scores.sort_values(ascending=False).index[:k]` — wrap in `list(...)`.

**Write your solution in `exercises/ex_3_2_recommend.py`**

In [ ]:
def recommend(person, k=3):
    pass   # replace this

# Tests
recs = recommend("SandeepGiri", 3)
assert len(recs) == 3
assert all(m in unrated_movies("SandeepGiri") for m in recs)
assert recs[0] == "Dangal"
print("SandeepGiri should watch:", recs)
print("Abhishek should watch:  ", recommend("Abhishek", 3))

You just built a working collaborative-filtering recommender on real data.
Try `recommend(...)` on a few more people — do the suggestions make sense
given who their taste twin is?

One honest limitation to notice: what would `recommend("YouJustJoined")` do
for a brand-new person with *zero* ratings? Nothing useful — there's no vector
to compare. This is the famous **cold-start problem**, and it's why every
streaming service pesters new users to "pick a few titles you like" before
showing recommendations.

## Part 4 (Bonus) — Two More Recommender Moves

### Exercise 4.1 — Ask your five best taste twins

`recommend()` let *everyone* vote, weighted by similarity. Real systems often
do something blunter and cheaper: find your **top 5 most similar people**,
pool their **top 5 favorite movies**, and drop anything you've already seen.
No scores, no weighted sums — just "what do people like me love?"

**Your task:** write three functions —

- `top5_similar_people(person)` — the 5 most similar people (by `sim`,
  excluding the person), most similar first.
- `top5_favorites(person)` — the person's 5 highest-**rated** movies. Use
  `ratings` (their real opinions), and `.dropna()` before sorting.
- `recommend_by_friends(person)` — pool the friends' favorites in order,
  remove movies `person` has already rated, and drop duplicates while keeping
  first appearance.

Then compare with `recommend(person, 3)` from Part 3 — do the two strategies
agree?

**Write your solution in `exercises/ex_4_1_top5_similar_people.py`**

In [ ]:
def top5_similar_people(person):
    pass  # replace this


def top5_favorites(person):
    pass  # replace this


def recommend_by_friends(person):
    pass  # replace this


# Tests
assert top5_similar_people("SandeepGiri") == ["Madhuri", "Stuti", "Prabha", "Aakash", "Gurpreet"]
assert len(top5_favorites("Abhishek")) == 5

friend_recs = recommend_by_friends("SandeepGiri")
assert set(friend_recs) == {"Mughal-e-Azam", "Dangal", "Yeh Jawaani Hai Deewani"}
assert all(m in unrated_movies("SandeepGiri") for m in friend_recs)
print("friends say:", friend_recs)
print("weighted votes said:", recommend("SandeepGiri", 3))

### Exercise 4.2 — Flip the matrix: similar *movies*

Everything we built compared **people** — the columns of `filled`. But the
same data read the other way compares **movies**: each *row* is a movie's
vector of 15 ratings, and two movies whose rows point the same way are liked
by the same people.

This powers the other famous recommendation sentence: *"customers who
watched this also watched…"* — no user profile needed at all.

**Your task:** repeat the Part 2 trick along the other axis:

1. Scale each **row** of `M = filled.values` to length 1. Row lengths:
   `np.sqrt((M ** 2).sum(axis=1, keepdims=True))` (`keepdims` makes the
   division broadcast row-wise).
2. `S_items = uMr @ uMr.T` — note the transpose moved to the *other* side:
   entry (i, j) is now row·row, movie i against movie j.
3. Wrap it: `msim = pd.DataFrame(S_items, index=filled.index, columns=filled.index)`.
4. Write `most_similar_movie(name)` (mirror of `most_similar`).

**What you invented:** **item-item collaborative filtering** — the variant
Amazon famously chose, because with millions of users and thousands of items,
item-item similarities are fewer, more stable, and reusable for every visitor.

**Write your solution in `exercises/ex_4_2_most_similar_movie.py`**

In [ ]:
uMr = ...      # replace this — unit rows of M
S_items = ...  # replace this
msim = ...     # replace this — labeled DataFrame

def most_similar_movie(name):
    pass  # replace this


# Tests
assert S_items.shape == (25, 25)
assert np.allclose(S_items, S_items.T)
assert np.allclose(np.diag(S_items), 1)
assert most_similar_movie("Sholay") == "3 Idiots"
for movie in ["Sholay", "Dangal", "Pathaan"]:
    print(f"people who liked {movie!r} also liked:",
          most_similar_movie(movie))

## Part 5 (Bonus) — Scaling to Netflix: MapReduce matrix multiplication

Our class matrix was 25×15. Netflix has ~300 million users and ~20,000 titles.
The user-similarity matrix alone would have $3\times10^8 \times 3\times10^8$
entries — no single machine can even *hold* that, let alone compute it.

Two observations rescue us:

1. **The matrix is almost entirely zeros.** A typical user rates a few dozen
   titles out of 20,000. Why store the zeros at all? Keep only the non-zero
   entries as `(row, col, value)` triples.
2. **Matrix multiplication decomposes into independent little jobs.** Recall:
   $(AB)_{ij} = \sum_k A_{ik} B_{kj}$. Each non-zero pair $A_{ik}, B_{kj}$
   contributes one product $A_{ik} \cdot B_{kj}$ to one output cell $(i, j)$ —
   and every contribution can be computed on a *different machine*, then
   summed up per cell. That "compute little contributions anywhere, then group
   and sum by key" pattern is called **MapReduce**, and it's how the giants do
   it.

**Your task:** write `sparse_matmul(A_triples, B_triples)` where each argument
is a list of `(row, col, value)` triples of the non-zero entries. Return a
dict mapping `(i, j) -> value` for the non-zero entries of the product.

Hints:
- For each `(i, k, a)` in A, find every `(k2, j, b)` in B with `k2 == k`, and
  add `a * b` into `out[(i, j)]`.
- A double loop over the triples works. For a cleaner version, first group B's
  triples into a dict keyed by row: `{k: [(j, b), ...]}` — that's the
  "group by key" half of MapReduce.
- `out.get((i, j), 0)` handles the first contribution to a cell.

In [ ]:
def sparse_matmul(A_triples, B_triples):
    pass   # replace this

# Tests — first a dense 2x2 you can verify by hand:
# [[1,2],[3,4]] @ [[5,6],[7,8]] = [[19,22],[43,50]]
A = [(0, 0, 1), (0, 1, 2), (1, 0, 3), (1, 1, 4)]
B = [(0, 0, 5), (0, 1, 6), (1, 0, 7), (1, 1, 8)]
assert sparse_matmul(A, B) == {(0, 0): 19, (0, 1): 22, (1, 0): 43, (1, 1): 50}

# Now truly sparse: two entries each, in a huge (implicit) matrix
A = [(0, 2, 5.0), (3, 1, 2.0)]
B = [(2, 0, 4.0), (1, 7, 3.0)]
assert sparse_matmul(A, B) == {(0, 0): 20.0, (3, 7): 6.0}
print("sparse_matmul works — zeros never stored, never multiplied")

## Summary — what you invented

| Idea | What it does |
|------|--------------|
| Dropping sparse raters | Fewer than 3 ratings tells us nothing about taste |
| Per-person min-max scaling | Puts generous and stingy raters on the same 0–1 scale |
| Fill NaN with the person's mean | A neutral "no opinion" value — invents no fake enthusiasm |
| Cosine similarity | Compares taste *direction*, ignoring rating magnitude |
| Unit vectors + `uM.T @ uM` | The entire 15×15 similarity table in one matrix multiplication |
| Similarity-weighted scoring | Your taste twins' opinions count the most |
| Friend-pooling | Top-5 similar people's top-5 favorites, minus what you've seen |
| Item-item similarity (`uMr @ uMr.T`) | "Customers who watched this also watched…" |
| Sparse MapReduce matmul | Store only non-zeros; split the multiplication into independent jobs |

This is **collaborative filtering** — recommending purely from the pattern of
who-likes-what, knowing nothing about the movies themselves (no genres, no
actors, no plots). The alternative, **content-based filtering**, compares
movie *features* instead; modern systems blend both.

And that `uM.T @ uM` trick — all-pairs similarity as a single matrix
product — will meet you again in the most unexpected place: it's the heart of
the *attention* mechanism inside transformers, where every word in a sentence
computes its similarity to every other word. Same math, different vectors.